In [ ]:
import os
os.getcwd()
os.chdir("C:\\path\\to\\CNC\\folder\\")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from scipy.signal import peak_prominences
import numpy as np
import datetime

path = 'datasets_pseudo/Threshold/'
plot_path = 'Plots/Final/'
date_str = datetime.datetime.now().strftime("%m%d%Y_%H%M%S")

names = os.listdir(path)

In [ ]:

# Choose column to detect peaks on
columns_to_predict = ['CURRENT|1', 'CURRENT|2', 'CURRENT|3', 'CURRENT|6']

# Moving average window size
window_sizes = [100,500, 750, 1000, 1250, 1500, 2000]

# Initially different peak sizes were tested out, however finally only threshold_high was considered. 
threshold_high, threshold_med, threshold_low = 2.5, 1.5, 1.2

os.makedirs("Results/", exist_ok=True)


In [ ]:
# The following tries to idenitify peaks in the CURRENT variable by using the TORQUE variable. 

results = []

for name in names:

    df_filtered = pd.read_csv(f"{path}\\{name}") 
    plot_limit = df_filtered.shape[0]

    for window_size in window_sizes:

        os.makedirs(f"{plot_path}\\MA\\{window_size}\\", exist_ok=True)

        for column in columns_to_predict:
            
            column_ = 'TORQUE|'+column[-1:]

            # moving average of the absolute value
            abs_col = f'{column_}_abs'
            df_filtered[abs_col] = df_filtered[column_].abs()
            
            df_filtered['MA_abs'] = df_filtered[abs_col].rolling(window=window_size, center=False).mean()

            # Peaks --> abs(value) > MA * threshold
            df_filtered['Peak_high'] = df_filtered[abs_col] > df_filtered['MA_abs'] * threshold_high
            df_filtered['Peak_med'] = df_filtered[abs_col] > df_filtered['MA_abs'] * threshold_med
            df_filtered['Peak_low'] = df_filtered[abs_col] > df_filtered['MA_abs'] * threshold_low

            # Evaluation
            accuracy = accuracy_score(df_filtered[f'{column}_Peak'], df_filtered['Peak_high'])
            precision = precision_score(df_filtered[f'{column}_Peak'], df_filtered['Peak_high'], zero_division=0)
            recall = recall_score(df_filtered[f'{column}_Peak'], df_filtered['Peak_high'], zero_division=0)
            f1 = f1_score(df_filtered[f'{column}_Peak'], df_filtered['Peak_high'], zero_division=0)
            n_peaks = df_filtered['Peak_high'].sum()
            peak_prominence = np.average(peak_prominences(df_filtered[column], df_filtered['Peak_high'], window_size)[0])

            results.append({
                'Dataset': name.replace(".csv", ""), 
                'Column': column,
                'Window': window_size,
                'Accuracy': accuracy,
                'Precision': precision,
                'Recall': recall,
                'F1': f1,
                'Detected_Peaks': n_peaks, 
                'Prominence': peak_prominence
                })
            
            subset = df_filtered.copy()

            #Plot
            plt.figure(figsize=(12, 6))
            plt.plot(subset[column], label='Current (Original)')
            plt.plot(subset.index[subset['Peak_high']], subset[column][subset['Peak_high']], 'rx', label='Peaks - High')

            plt.title(f"Absolute-Magnitude Peak Detection in {column}")
            plt.xlabel("Index")
            plt.ylabel("Current")
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'{plot_path}\\MA\\{window_size}\\MA_{column.replace("|", "")}_{name.replace("threshold_", "")}.png', dpi=300, bbox_inches='tight')
            # plt.show()   #uncomment if required to plot in notebook 

# Save to DataFrame
results_df = pd.DataFrame(results)
results_df.sort_values(by='F1', ascending=False, inplace=True)
    
display(results_df)

results_df.to_csv(f"Results/MA_results_{date_str}.csv")
